# Chapter 18: Building a CSV Data Explorer Dashboard

Companion notebook for *Python by Curiosity*. Every code listing from the chapter is
here, in the order it appears in the book, under the same title. Run the cells from
top to bottom: some listings use variables or functions defined in earlier cells.

The same listings are also available as separate scripts in `codes/listings/ch18_data_explorer/`.

**These listings open desktop windows (Tkinter).** Run the notebook on your own
computer (Jupyter or VS Code), not in an online service such as Google Colab. Each
cell waits until you close its window; close it to move on to the next cell.

In [ ]:
# Run the listings from the repository root so that paths such as
# data/grades.csv and figures/trophy.png work.
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("Working folder:", os.getcwd())

### Code 18.1: File picker dialog with extension filtering

In [ ]:
from tkinter import filedialog

filepath = filedialog.askopenfilename(
    title="Select Scientific CSV Dataset",
    filetypes=[("CSV Files", "*.csv"), ("Text Files", "*.txt"), ("All Files", "*.*")]
)

if filepath:
    print(f"Loading file: {filepath}")
else:
    print("User cancelled file selection.")

### Code 18.2: Extracting numeric values from a selected column

In [ ]:
def extract_column_data(rows, col_index):
    numbers = []
    # Skip header row (rows[0])
    for row in rows[1:]:
        if len(row) > col_index:
            try:
                val = float(row[col_index])
                numbers.append(val)
            except ValueError:
                continue # Skip non-numeric values or missing entries
    return numbers

### 4. Vectorized NumPy Statistical Computation

*Shown in the book as a fragment or a deliberate mistake; on its own it stops with NameError.* The code is shown here for reference:

```python
data_arr = np.array(numbers)
mean_val = np.mean(data_arr)
median_val = np.median(data_arr)
std_val = np.std(data_arr, ddof=1)
min_val = np.min(data_arr)
max_val = np.max(data_arr)
```

### Code 18.3: Scientific CSV Data Explorer Dashboard

In [ ]:
import tkinter as tk
from tkinter import ttk, filedialog, messagebox
import numpy as np
import matplotlib.pyplot as plt
import csv
import os

class DataExplorerApp:
    def __init__(self, root):
        self.root = root
        self.root.title("CSV Data Explorer & Statistical Workbench")
        self.root.geometry("560x480")
        self.root.configure(bg="#f4f6f9")
        
        self.rows = []
        self.headers = []
        
        self.build_ui()

    def build_ui(self):
        # Header Banner
        hdr = tk.Label(self.root, text="SCIENTIFIC CSV DATA EXPLORER & WORKBENCH", 
                       font=("Arial", 11, "bold"), bg="#2c3e50", fg="white", pady=6)
        hdr.pack(fill="x")

        # BLOCK 1: File Ingestion Header
        f_top = tk.Frame(self.root, bg="#f4f6f9", padx=10, pady=5)
        f_top.pack(fill="x")

        btn_load = tk.Button(f_top, text="[Open CSV File]", font=("Arial", 9, "bold"), 
                            bg="#2980b9", fg="white", padx=8, pady=3, command=self.load_csv)
        btn_load.pack(side="left")

        self.lbl_file = tk.Label(f_top, text="No file loaded", font=("Arial", 9, "bold"), 
                                 fg="gray", bg="#f4f6f9")
        self.lbl_file.pack(side="left", padx=10)

        # BLOCK 2: Scrollable Raw Data Text View
        f_content = tk.Frame(self.root, bg="#f4f6f9", padx=10, pady=2)
        f_content.pack(fill="both", expand=True)

        sb = tk.Scrollbar(f_content)
        sb.pack(side="right", fill="y")

        self.txt = tk.Text(f_content, wrap="none", yscrollcommand=sb.set, 
                           font=("Courier", 9), height=10, bg="#ffffff", fg="#2c3e50", bd=1, relief="solid")
        self.txt.pack(side="left", fill="both", expand=True)
        sb.config(command=self.txt.yview)

        # BLOCK 3: Dynamic Column & Filter Controls
        f_ctrl = tk.Frame(self.root, bg="#eaeded", bd=1, relief="groove", padx=10, pady=6)
        f_ctrl.pack(fill="x", padx=10, pady=5)

        tk.Label(f_ctrl, text="Target Column:", font=("Arial", 9, "bold"), bg="#eaeded").grid(row=0, column=0, sticky="w")
        
        self.combo_cols = ttk.Combobox(f_ctrl, state="readonly", width=20)
        self.combo_cols.grid(row=0, column=1, padx=5, sticky="w")
        self.combo_cols.bind("<<ComboboxSelected>>", self.analyze_selected_column)

        btn_plot = tk.Button(f_ctrl, text="[Plot Column Histogram]", font=("Arial", 9, "bold"), 
                             bg="#8e44ad", fg="white", padx=8, pady=2, command=self.plot_histogram)
        btn_plot.grid(row=0, column=2, padx=12)

        # BLOCK 4: Summary Statistics Card
        self.stats_card = tk.LabelFrame(self.root, text="NumPy Summary Statistics", 
                                       font=("Arial", 9, "bold"), bg="#ffffff", fg="#2c3e50", padx=10, pady=6)
        self.stats_card.pack(fill="x", padx=10, pady=5)

        self.lbl_mean = tk.Label(self.stats_card, text="Mean: N/A", font=("Arial", 9, "bold"), fg="#2980b9", bg="#ffffff")
        self.lbl_mean.grid(row=0, column=0, padx=10, pady=3)
        
        self.lbl_median = tk.Label(self.stats_card, text="Median: N/A", font=("Arial", 9, "bold"), fg="#27ae60", bg="#ffffff")
        self.lbl_median.grid(row=0, column=1, padx=10, pady=3)
        
        self.lbl_std = tk.Label(self.stats_card, text="Std Dev: N/A", font=("Arial", 9, "bold"), fg="#e67e22", bg="#ffffff")
        self.lbl_std.grid(row=0, column=2, padx=10, pady=3)

        self.lbl_min = tk.Label(self.stats_card, text="Min: N/A", font=("Arial", 9, "bold"), fg="#7f8c8d", bg="#ffffff")
        self.lbl_min.grid(row=0, column=3, padx=10, pady=3)

        self.lbl_max = tk.Label(self.stats_card, text="Max: N/A", font=("Arial", 9, "bold"), fg="#e74c3c", bg="#ffffff")
        self.lbl_max.grid(row=0, column=4, padx=10, pady=3)

        # BLOCK 5: Status Bar
        self.lbl_status = tk.Label(self.root, text="Status: Ready | Load a CSV file to begin", 
                                   bd=1, relief="sunken", anchor="w", font=("Arial", 8, "italic"), bg="#e0e0e0", fg="#333333")
        self.lbl_status.pack(side="bottom", fill="x")

    def load_csv(self):
        filepath = filedialog.askopenfilename(filetypes=[("CSV Files", "*.csv"), ("All Files", "*.*")])
        if not filepath:
            return
        
        filename = os.path.basename(filepath)
        self.lbl_file.config(text=f"Loaded: {filename}", fg="#27ae60")
        self.txt.delete("1.0", tk.END)

        try:
            with open(filepath, "r", newline="", encoding="utf-8") as file:
                reader = csv.reader(file)
                self.rows = list(reader)

            if not self.rows:
                raise ValueError("The selected CSV file is empty.")

            self.headers = self.rows[0]
            self.combo_cols['values'] = [f"{h} (Col {i+1})" for i, h in enumerate(self.headers)]
            if self.headers:
                self.combo_cols.current(0)

            # Insert formatted rows into text widget
            formatted_txt = ""
            for row in self.rows:
                formatted_txt += ",  ".join(row) + "\n"
            self.txt.insert(tk.END, formatted_txt)

            self.analyze_selected_column()
        except Exception as err:
            messagebox.showerror("File Error", f"Could not parse CSV:\n{err}")
            self.lbl_file.config(text="Error loading file", fg="red")

    def analyze_selected_column(self, event=None):
        if not self.rows or len(self.rows) < 2:
            return

        col_idx = self.combo_cols.current()
        if col_idx < 0:
            col_idx = 0

        numbers = []
        for row in self.rows[1:]:
            if len(row) > col_idx:
                try:
                    numbers.append(float(row[col_idx]))
                except ValueError:
                    continue

        if numbers:
            arr = np.array(numbers)
            self.lbl_mean.config(text=f"Mean: {np.mean(arr):.2f}")
            self.lbl_median.config(text=f"Median: {np.median(arr):.2f}")
            self.lbl_std.config(text=f"Std Dev: {np.std(arr, ddof=1):.2f}")
            self.lbl_min.config(text=f"Min: {np.min(arr):.2f}")
            self.lbl_max.config(text=f"Max: {np.max(arr):.2f}")
            self.lbl_status.config(text=f"Status: Analyzed {len(numbers)} numeric rows in '{self.headers[col_idx]}'")
        else:
            self.lbl_mean.config(text="Mean: N/A")
            self.lbl_median.config(text="Median: N/A")
            self.lbl_std.config(text="Std Dev: N/A")
            self.lbl_min.config(text="Min: N/A")
            self.lbl_max.config(text="Max: N/A")
            self.lbl_status.config(text="Status: Selected column contains no numeric data.")

    def plot_histogram(self):
        if not self.rows or len(self.rows) < 2:
            messagebox.showwarning("Warning", "Please load a CSV file first!")
            return

        col_idx = self.combo_cols.current()
        numbers = []                   # same rule as analyze_selected_column
        for row in self.rows[1:]:
            if len(row) > col_idx:
                try:
                    numbers.append(float(row[col_idx]))
                except ValueError:
                    continue

        if not numbers:
            messagebox.showwarning("Warning", "Selected column contains no valid numeric data to plot!")
            return

        plt.figure(figsize=(6, 4))
        plt.hist(numbers, bins=15, color="#2980b9", edgecolor="white")
        plt.title(f"Distribution Histogram: {self.headers[col_idx]}")
        plt.xlabel(self.headers[col_idx])
        plt.ylabel("Frequency")
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

if __name__ == "__main__":
    root = tk.Tk()
    app = DataExplorerApp(root)
    root.mainloop()

### Worked Example 18.1: Sensor Log Analyzer with Outlier Filtering

In [ ]:
import tkinter as tk
from tkinter import messagebox
import numpy as np

class SensorAnalyzerApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Sensor Log Outlier Analyzer")
        self.root.geometry("450x320")
        self.root.configure(padx=15, pady=15)
        
        # Simulated temperature sensor telemetry
        self.data = np.array([21.4, 21.8, 22.1, 21.9, 28.5, 22.0, 21.7, 31.2, 22.3, 21.5])
        
        tk.Label(root, text="SENSOR TELEMETRY OUTLIER FILTER", font=("Arial", 11, "bold"), fg="#2c3e50").pack(pady=(0, 10))
        
        # Raw Data Display Box
        self.txt_data = tk.Text(root, height=4, font=("Courier", 9))
        self.txt_data.insert(tk.END, ", ".join(f"{v:.1f}" for v in self.data))
        self.txt_data.pack(fill="x", pady=5)
        
        # Threshold Input
        ctrl_frame = tk.Frame(root)
        ctrl_frame.pack(fill="x", pady=8)
        
        tk.Label(ctrl_frame, text="Outlier Threshold (deg C):", font=("Arial", 10)).pack(side="left")
        self.ent_thresh = tk.Entry(ctrl_frame, width=8, font=("Arial", 10))
        self.ent_thresh.insert(0, "25.0")
        self.ent_thresh.pack(side="left", padx=10)
        
        btn_filter = tk.Button(ctrl_frame, text="Filter Outliers", font=("Arial", 9, "bold"),
                               bg="#e67e22", fg="white", command=self.apply_filter)
        btn_filter.pack(side="left")
        
        # Results Card
        self.lbl_result = tk.Label(root, text="Results: Click 'Filter Outliers' to analyze.",
                                   font=("Arial", 10), justify="left", bg="#ecf0f1", padx=10, pady=10, relief="solid", bd=1)
        self.lbl_result.pack(fill="both", expand=True, pady=10)

    def apply_filter(self):
        try:
            thresh = float(self.ent_thresh.get())
            outliers = self.data[self.data > thresh]
            normal = self.data[self.data <= thresh]
            
            res_str = (
                f"Total Readings: {len(self.data)}\n"
                f"Normal Readings: {len(normal)} (Mean: {np.mean(normal):.2f} C)\n"
                f"Outliers Detected: {len(outliers)} (Mean: {np.mean(outliers) if len(outliers) else 0.0:.2f} C)\n"
                f"Outlier Values: {list(outliers)}"
            )
            self.lbl_result.config(text=res_str)
        except ValueError:
            messagebox.showerror("Error", "Please enter a valid numeric threshold.")

if __name__ == "__main__":
    root = tk.Tk()
    app = SensorAnalyzerApp(root)
    root.mainloop()

### Worked Example 18.2: Two-Variable Correlation Dashboard

In [ ]:
import tkinter as tk
import numpy as np
import matplotlib.pyplot as plt

class CorrelationDashboardApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Two-Variable Correlation Dashboard")
        self.root.geometry("420x220")
        self.root.configure(padx=15, pady=15)
        
        # Paired datasets: Study Hours (X) vs Exam Score (Y)
        self.x = np.array([2.5, 5.1, 3.2, 8.5, 3.5, 1.5, 9.2, 5.5, 8.3, 2.7])
        self.y = np.array([30,   54,  41,  92,  45,  22,  98,  60,  88,  33])
        
        tk.Label(root, text="BIVARIATE CORRELATION WORKBENCH", font=("Arial", 11, "bold"), fg="#2c3e50").pack(pady=(0, 10))
        
        r = np.corrcoef(self.x, self.y)[0, 1]
        if abs(r) >= 0.7:
            strength = "Strong"
        elif abs(r) >= 0.4:
            strength = "Moderate"
        else:
            strength = "Weak"
        direction = "Positive" if r > 0 else "Negative"
        self.lbl_stats = tk.Label(root, text=f"Pearson Correlation (r): {r:.4f}\nStrength: {strength} {direction} Linear Relationship",
                                  font=("Arial", 10, "bold"), fg="#27ae60", bg="#ecf0f1", padx=10, pady=8, bd=1, relief="ridge")
        self.lbl_stats.pack(fill="x", pady=10)
        
        btn_plot = tk.Button(root, text="Plot Scatter & Trendline", font=("Arial", 10, "bold"),
                             bg="#2980b9", fg="white", pady=6, command=self.plot_scatter)
        btn_plot.pack(fill="x", pady=5)

    def plot_scatter(self):
        m, c = np.polyfit(self.x, self.y, 1)  # Linear regression fit
        plt.figure(figsize=(6, 4))
        plt.scatter(self.x, self.y, color="#e74c3c", s=50, label="Observations")
        plt.plot(self.x, m * self.x + c, color="#2980b9", lw=2, label=f"Fit: y = {m:.2f}x + {c:.2f}")
        plt.title("Study Hours vs. Exam Score Correlation", fontweight="bold")
        plt.xlabel("Study Hours (X)")
        plt.ylabel("Exam Score (Y)")
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.tight_layout()
        plt.show()

if __name__ == "__main__":
    root = tk.Tk()
    app = CorrelationDashboardApp(root)
    root.mainloop()